# MAVLink mesajını kodlayın ve çözün

Yer istasyonunun her saniye gönderdiği HEARTBEAT mesajını baytlara çevirip geri çözüyoruz. Bir baytı bozunca CRC denetimi mesajı reddeder: telemetrideki "bozuk paket" sayacı tam olarak bu.

Kodu değiştirip hücreyi yeniden çalıştırın (Shift+Enter). Konunun tamamı: [softivation.com/egitimler/python-ile-sifirdan-iha-kontrolu-egitimi](https://softivation.com/egitimler/python-ile-sifirdan-iha-kontrolu-egitimi)

In [ ]:
# pymavlink Colab'da hazır gelmez; sabit sürümü kurulur
%pip install -q pymavlink==2.4.50

In [ ]:
# HEARTBEAT: yer istasyonunun (GCS) her saniye gönderdiği kalp atışı mesajı
from pymavlink.dialects.v20 import ardupilotmega as mavlink

class Tampon:
    def __init__(self):
        self.veri = b""
    def write(self, b):
        self.veri += b

t = Tampon()
gcs = mavlink.MAVLink(t, srcSystem=255, srcComponent=190)
gcs.heartbeat_send(mavlink.MAV_TYPE_GCS, mavlink.MAV_AUTOPILOT_INVALID, 0, 0, 0)
print("Baytlar :", t.veri.hex(" "))
print("Uzunluk :", len(t.veri), "bayt (başlık 10 + yük 9 + CRC 2)")

m = mavlink.MAVLink(None).parse_buffer(t.veri)[0]
print("Çözülen :", m.get_type(), "| sistem", m.get_srcSystem(), "| bileşen", m.get_srcComponent())

# Bir baytı bozun: CRC tutmaz, mesaj kabul edilmez
bozuk = bytearray(t.veri)
bozuk[12] ^= 0xFF
cozucu = mavlink.MAVLink(None)
try:
    sonuc = cozucu.parse_buffer(bytes(bozuk))
    print("Bozuk paket:", sonuc)
except mavlink.MAVError as e:
    print("Bozuk paket reddedildi:", e)